In [1]:
import json

In [2]:
file_path = "../../data/conv-trs/ecir-2026/human-eval/surveys_collection_raw.json"

with open(file_path, 'r') as f:
    og_data = json.load(f)
og_data[0]

{'timestamp': '2025-10-18T22:25:41.779Z',
 'completed': False,
 'prolificId': 'Ewald',
 'currentPageIndex': 102,
 'flatAnswers': {'ranking_q32_dim_popularity_mix_qid_0': 4,
  'ranking_q57_dim_popularity_mix_qid_0': 4,
  'ranking_q31_dim_popularity_mix_qid_0': 4,
  'ranking_q65_dim_popularity_mix_qid_0': 3,
  'ranking_q25_dim_relevance_match': 2,
  'ranking_q99_dim_sustainability_qid_0': 3,
  'ranking_q30_dim_sustainability_qid_0': 4,
  'ranking_q79_dim_popularity_mix_qid_0': 4,
  'ranking_q45_dim_sustainability_qid_0': 4,
  'ranking_q21_dim_sustainability_qid_0': 4,
  'ranking_q92_dim_sustainability_qid_0': 3,
  'ranking_q43_dim_popularity_mix_qid_0': 2,
  'ranking_q80_dim_diversity_qid_0': 3,
  'ranking_q62_dim_popularity_mix_qid_0': 2,
  'ranking_q68_dim_sustainability_qid_0': 3,
  'ranking_q30_dim_relevance_match': 4,
  'ranking_q51_dim_popularity_mix_qid_0': 2,
  'ranking_q67_dim_popularity_mix_qid_0': 2,
  'ranking_q92_dim_diversity_qid_0': 4,
  'ranking_q46_dim_relevance_match': 

In [10]:
og_data[2]

{'timestamp': '2025-10-19T11:33:23.637Z',
 'completed': False,
 'prolificId': 'Wolfgang',
 'currentPageIndex': 102,
 'flatAnswers': {'ranking_q66_dim_diversity_qid_0_not_sure': True,
  'ranking_q32_dim_popularity_mix_qid_0': 3,
  'ranking_q57_dim_popularity_mix_qid_0': 2,
  'ranking_q31_dim_popularity_mix_qid_0': 4,
  'ranking_q65_dim_popularity_mix_qid_0': 4,
  'ranking_q25_dim_relevance_match': 2,
  'ranking_q99_dim_sustainability_qid_0': 4,
  'ranking_q43_dim_popularity_mix_qid_0': 3,
  'ranking_q79_dim_popularity_mix_qid_0': 4,
  'ranking_q80_dim_diversity_qid_0': 2,
  'ranking_q21_dim_sustainability_qid_0': 1,
  'ranking_q92_dim_sustainability_qid_0': 3,
  'ranking_q31_dim_sustainability_qid_0_not_sure': True,
  'ranking_q45_dim_sustainability_qid_0': 2,
  'ranking_q62_dim_popularity_mix_qid_0': 3,
  'ranking_q30_dim_sustainability_qid_0': 2,
  'ranking_q68_dim_sustainability_qid_0': 2,
  'ranking_q34_dim_sustainability_qid_0_not_sure': True,
  'ranking_q92_dim_diversity_qid_0': 3

In [3]:
import re
import copy
import json

# Your JSON data
data = og_data[0]  # replace this

# Deep copy for safety
answers = copy.deepcopy(data["answers"]["ranking-eval"])
flat = data["flatAnswers"]

# Regex pattern to extract q-number and dimension
pattern = re.compile(r"ranking_q(\d+)_dim_([a-zA-Z_]+)")

# Define comparison map (converted to labels with A/B)
LABEL_MAP = {
    -2: "Much more A than B",
    -1: "Slightly more A than B",
    0: "About the same",
    1: "Slightly more B than A",
    2: "Much more B than A",
    -3: "Not sure / Don’t know"
}

# Process all flat answers
for key, value in flat.items():
    match = pattern.search(key)
    if not match:
        continue

    q_index = int(match.group(1)) - 1  # e.g., q1 → index 0
    dim = match.group(2)

    # Skip invalid indices
    if q_index >= len(answers):
        continue

    # Ensure dimension exists
    if dim not in answers[q_index]:
        answers[q_index][dim] = {}

    # Handle "not sure" fields
    if key.endswith("_not_sure"):
        answers[q_index][dim]["not_sure"] = bool(value)
        if value:  # if they said "not sure", assign that label
            answers[q_index][dim]["label"] = LABEL_MAP[-3]
            answers[q_index][dim]["value"] = -3
    else:
        # Assign numeric value and label
        answers[q_index][dim]["value"] = value
        answers[q_index][dim]["label"] = LABEL_MAP.get(value, "Unknown")
        answers[q_index][dim].setdefault("not_sure", False)

# Update JSON
data["answers"]["ranking-eval"] = answers
data["prolificId"] = "Ewald_cleaned"

# Show first few updated answers for verification
print(json.dumps(data["answers"]["ranking-eval"][:3], indent=2))


[
  {
    "query_id": "c_p_68_pop_medium_sustainable",
    "is_practice_query": true,
    "query": "Budget-friendly European city break in April.",
    "popularity_mix_qid_": {
      "value": 4,
      "label": "Unknown",
      "not_sure": false
    },
    "relevance_match": {
      "value": 4,
      "label": "Unknown",
      "not_sure": false
    },
    "diversity_qid_": {
      "value": 3,
      "label": "Unknown",
      "not_sure": false
    },
    "sustainability_qid_": {
      "value": 4,
      "label": "Unknown",
      "not_sure": false
    }
  },
  {
    "query_id": "c_p_196_pop_high_sustainable",
    "is_practice_query": false,
    "query": "Cheap European city break in October with museums and nightlife.",
    "sustainability_qid_": {
      "value": 2,
      "label": "Much more B than A",
      "not_sure": false
    },
    "diversity_qid_": {
      "value": 2,
      "label": "Much more B than A",
      "not_sure": false
    },
    "popularity_mix_qid_": {
      "value": 4,
    

In [4]:
data

{'timestamp': '2025-10-18T22:25:41.779Z',
 'completed': False,
 'prolificId': 'Ewald_cleaned',
 'currentPageIndex': 102,
 'flatAnswers': {'ranking_q32_dim_popularity_mix_qid_0': 4,
  'ranking_q57_dim_popularity_mix_qid_0': 4,
  'ranking_q31_dim_popularity_mix_qid_0': 4,
  'ranking_q65_dim_popularity_mix_qid_0': 3,
  'ranking_q25_dim_relevance_match': 2,
  'ranking_q99_dim_sustainability_qid_0': 3,
  'ranking_q30_dim_sustainability_qid_0': 4,
  'ranking_q79_dim_popularity_mix_qid_0': 4,
  'ranking_q45_dim_sustainability_qid_0': 4,
  'ranking_q21_dim_sustainability_qid_0': 4,
  'ranking_q92_dim_sustainability_qid_0': 3,
  'ranking_q43_dim_popularity_mix_qid_0': 2,
  'ranking_q80_dim_diversity_qid_0': 3,
  'ranking_q62_dim_popularity_mix_qid_0': 2,
  'ranking_q68_dim_sustainability_qid_0': 3,
  'ranking_q30_dim_relevance_match': 4,
  'ranking_q51_dim_popularity_mix_qid_0': 2,
  'ranking_q67_dim_popularity_mix_qid_0': 2,
  'ranking_q92_dim_diversity_qid_0': 4,
  'ranking_q46_dim_relevance_

In [5]:
len(og_data)

3

In [6]:
len(og_data)

3

In [7]:
og_data[2]["prolificId"]

'Wolfgang'

In [8]:
with open("../../data/conv-trs/ecir-2026/human-eval/surveys_collection_v2.json", "w") as outfile:
    json.dump(og_data, outfile, indent=4)

In [9]:
len(og_data)

3